## ML Project (END SEM)

We start by importing all the required libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

Reading the dataset

In [ ]:
df = pd.read_csv("StudentPerformanceFactors.csv")
df.head()


### Basic Analysis of Dataset

In [ ]:
# data on all features
df.info()

In [ ]:
# statistics of all features
df.describe()

### Checking for null values

In [ ]:
df.isnull().sum()

We can see that Teacher_Quality,Parental_Education_Level,Distance_from_Home columns have null values all are categorical columns so we will replace by mode

In [ ]:
for col in df.columns:
    if df[col].dtype in ['int64', 'float64']: 
        mean_value = df[col].mean()
        df[col].fillna(mean_value, inplace=True)
    else: 
        mode_value = df[col].mode()[0]
        df[col].fillna(mode_value, inplace=True)
df.isnull().sum()


In [ ]:
df.describe()

### Checking existence of duplicates:

In [ ]:
df = df.drop_duplicates()
df.shape

In [ ]:
num_cols=[col for col in df.columns if df[col].dtype in ['int64', 'float64']]

### Outlier Detection and Removal

In [ ]:
plt.figure(figsize=(12,6))
cols = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] and col != 'Exam_Score']
sns.boxplot(data=df[cols])
plt.xticks(rotation=90)
plt.title("Boxplot for Outlier Detection")
plt.show()

We notice a few outliers in the boxplot. Since the outliers are quite far away from the IQR, they can be removed.

In [ ]:
Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1

ol = ((df[num_cols] < (Q1 - 1.5 * IQR)) | (df[num_cols] > (Q3 + 1.5 * IQR)))
ol_iqr = ol.any(axis=1)
print(f'No. of outliers: {np.sum(ol_iqr)}')
df = df[~ol_iqr]

In [ ]:
plt.figure(figsize=(12,6))
cols = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] and col != 'Exam_Score']
sns.boxplot(data=df[cols])
plt.xticks(rotation=90)
plt.title("Boxplot for Outlier Detection")
plt.show()

### Exploratory Data Analysis

#### Univariate Analysis

#### Numerical Features

In [ ]:
colors = ['k', 'r', 'c', 'g']
num_cols = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] and col not in ['Tutoring_Sessions','Physical_Activity','Sleep_Hours'] ]
plt.figure(figsize=(20, 25))
for i, col in enumerate(num_cols):
    plt.subplot(4, 3, i + 1)
    plt.title(' '.join(col.split('_')).title())
    sns.histplot(df[col], kde=True, color=colors[i % len(colors)])
plt.show()

In [ ]:
import math

colors = ['k', 'g', 'r', 'c']
cat_cols = ['Tutoring_Sessions','Physical_Activity','Sleep_Hours']
n_cols = 2 
n_rows = math.ceil(len(cat_cols) / n_cols) 

plt.figure(figsize=(15, n_rows * 5)) 

for i, col in enumerate(cat_cols):
    plt.subplot(n_rows, n_cols, i + 1)
    plt.title(' '.join(col.split('_')).title())
    sns.countplot(x=df[col], color=colors[i % len(colors)])
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

#### Categorical Features

In [ ]:
colors = ['k', 'g', 'r', 'c']
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
n_cols = 2 
n_rows = math.ceil(len(cat_cols) / n_cols) 

plt.figure(figsize=(15, n_rows * 5)) 

for i, col in enumerate(cat_cols):
    plt.subplot(n_rows, n_cols, i + 1)
    plt.title(' '.join(col.split('_')).title())
    sns.countplot(x=df[col], color=colors[i % len(colors)])
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

##### Distribution of classes of Target

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Exam_Score'], kde=True, color='skyblue', bins=20)
plt.title('Distribution of Exam Score')
plt.xlabel('Exam_Score')
plt.ylabel('Frequency')
plt.show()


#### Bivariate Analysis

#### Numerical Features

In [ ]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols.remove('Exam_Score')  # remove target

plt.figure(figsize=(15, len(num_cols)*4))

for i, col in enumerate(num_cols):
    plt.subplot(len(num_cols), 1, i + 1)
    sns.scatterplot(x=df[col], y=df['Exam_Score'])
    plt.title(f'{col} vs Exam Score')
    plt.xlabel(col)
    plt.ylabel('Exam Score')

plt.tight_layout()
plt.show()


#### Categorical Features

In [ ]:
palette = 'Set2'

n_cols = 1
n_rows = math.ceil(len(cat_cols) / n_cols)

plt.figure(figsize=(15, n_rows * 5))

for i, col in enumerate(cat_cols):
    plt.subplot(n_rows, n_cols, i + 1)
    sns.violinplot(x=col, y='Exam_Score', data=df, palette=palette)
    plt.title(" ".join(col.split('_')).title())
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.preprocessing import LabelEncoder
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print("Categorical Columns:", len(cat_cols))
num_cols = df.select_dtypes(include=['number']).columns.tolist()
print("Numerical Columns:", len(num_cols))
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

#### Multivariate Analysis

We plot the heatmap for the dataset to see the correlation between features

In [ ]:
plt.figure(figsize=(20,20))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

we are selecting all the features as maximum correlation is 0.58

In [ ]:
# standardization
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
for col in num_cols:
    df[col] = sc.fit_transform(df[col].values.reshape(-1, 1))
df.head()

## MODEL TRAINING

In [ ]:
X = df.drop('Exam_Score',axis=1)
y = df['Exam_Score']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2,random_state=42)

### Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection  import cross_val_score

lr = LinearRegression()
lr.fit(X_train, y_train)

y_train_pred_linear = lr.predict(X_train)
y_test_pred_linear = lr.predict(X_test)

train_acc = r2_score(y_train, y_train_pred_linear)
test_acc = r2_score(y_test, y_test_pred_linear)

print(f"r2 score for train: {train_acc}")
print(f"r2 score for test: {test_acc}")
print(f"mse for train: {mean_squared_error(y_train, y_train_pred_linear)}")
print(f"mse for test: {mean_squared_error(y_test, y_test_pred_linear)}")
cv_scores = cross_val_score(lr, X_train, y_train, cv=5, scoring='r2')
print(f"CV scores:{cv_scores}")
print(f"Average CV score:{cv_scores.mean()}")

#### Actual v/s Predicted

In [ ]:
plt.figure(figsize=(7,5))

plt.scatter(y_test, y_test_pred_linear)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         linewidth=2)

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted (Test Set)")
plt.show()

The points do not lie exactly on the diagonal line. We can see that there is vertical spread across the diagonal line, the error is also consistent with the trend and spread is higher for larger values, but we can say that this model fits the data reasonably well the model. It is not perfect but it is an acceptable model.

### Polynomial Regression

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(5)
X_train_poly=poly.fit_transform(X_train)
X_test_poly=poly.transform(X_test)

In [ ]:
plr = LinearRegression()
plr.fit(X_train_poly, y_train)

y_train_pred_poly = plr.predict(X_train_poly)
y_test_pred_poly = plr.predict(X_test_poly)

train_acc = r2_score(y_train, y_train_pred_poly)
test_acc = r2_score(y_test, y_test_pred_poly)

print(f"r2 score for train: {train_acc}")
print(f"r2 score for test: {test_acc}")
print(f"mse for train: {mean_squared_error(y_train, y_train_pred_poly)}")
print(f"mse for test: {mean_squared_error(y_test, y_test_pred_poly)}")
cv_scores = cross_val_score(plr, X_train, y_train, cv=5, scoring='r2')
print(f"CV scores:{cv_scores}")
print(f"Average CV score:{cv_scores.mean()}")

#### Hyperparameter Tuning

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    ('poly', PolynomialFeatures()), 
    ('linear', LinearRegression()) 
])

param_grid = {
    'poly__degree': np.arange(1, 6),
}

gscv = GridSearchCV(estimator=pipeline, param_grid=param_grid, scoring='r2', cv=5, verbose=1, n_jobs=-1)
gscv.fit(X_train, y_train)

print(f"Best parameters found: {gscv.best_params_}")
print(f"Best score: {gscv.best_score_}")
best_poly_model = gscv.best_estimator_

test_r2 = best_poly_model.score(X_test, y_test)
y_test_pred_poly = best_poly_model.predict(X_test)
y_train_pred_poly = best_poly_model.predict(X_train)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_poly))

print(f"Test R2: {test_r2}")
print(f"Test RMSE: {test_rmse}")

#### Learning Curve

In [ ]:
from sklearn.model_selection import learning_curve

best_params = best_poly_model.get_params()
best_degree = best_params['poly__degree']   

poly_final = Pipeline([
    ('poly', PolynomialFeatures(degree=best_degree, include_bias=False)),
    ('linear', LinearRegression())
])

train_sizes, train_scores, test_scores = learning_curve(estimator=poly_final, X=X_train, y=y_train, cv=5, scoring='r2', train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1, verbose=1)

train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, marker='o', label='Training R2')
plt.plot(train_sizes, test_mean, marker='s', label='Validation R2')

plt.title(f"Learning Curve for Tuned Polynomial Regression (Degree = {best_degree})")
plt.xlabel("Training Samples")
plt.ylabel("R2 Score")
plt.grid(True)
plt.legend()
plt.show()

#### Actual v/s Predicted

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(y_test,y_test_pred_poly)
plt.plot([y_test.min(), y_test.max()], 
         [y_test.min(), y_test.max()],
         linewidth=2)

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted (Best Polynomial Model)")
plt.show()

Polynomial Regression is fitting the pattern better than linear Regression as we can see that the predictions are more accurate than before. But it is slightly overfitting. However, the overall performance is good. The hyperparameter tuning has improved the model very significantly, the new model is extremely accurate.

### LASSO

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score

lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)

y_train_pred_lasso = lasso.predict(X_train)
y_test_pred_lasso = lasso.predict(X_test)

train_acc = r2_score(y_train, y_train_pred_lasso)
test_acc = r2_score(y_test, y_test_pred_lasso)

print(f"r2 score for train: {train_acc}")
print(f"r2 score for test: {test_acc}")
print(f"mse for train: {mean_squared_error(y_train, y_train_pred_lasso)}")
print(f"mse for test: {mean_squared_error(y_test, y_test_pred_lasso)}")
cv_scores = cross_val_score(lasso, X_train, y_train, cv=5, scoring='r2')
print(f"CV scores:{cv_scores}")
print(f"Average CV score:{cv_scores.mean()}")

#### Hyperparameter Tuning

In [ ]:
# Hyperparameter grid
param_grid = {
    'alpha': np.logspace(-4, 2, 20)   # 0.0001 → 100
}

# GridSearchCV
grid = GridSearchCV(
    estimator=Lasso(max_iter=5000),
    param_grid=param_grid,
    scoring='r2',
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV R2 score:", grid.best_score_)

# Best model
best_lasso = grid.best_estimator_

# Predictions
y_train_pred_lasso = best_lasso.predict(X_train)
y_test_pred_lasso = best_lasso.predict(X_test)

print("Train R2:", r2_score(y_train, y_train_pred_lasso))
print("Test R2:", r2_score(y_test, y_test_pred_lasso))
print("Train MSE:", mean_squared_error(y_train, y_train_pred_lasso))
print("Test MSE:", mean_squared_error(y_test, y_test_pred_lasso))

#### Actual v/s Predicted

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,5))
plt.scatter(y_test, y_test_pred_lasso)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         linewidth=2)

plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Lasso Regression: Actual vs Predicted")
plt.show()

We can say that the LASSO model has improved after tuning but still it is not perfect.

### Ridge

In [ ]:
from sklearn.linear_model import Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

y_train_pred_ridge = ridge.predict(X_train)
y_test_pred_ridge = ridge.predict(X_test)

mse = mean_squared_error(y_test, y_test_pred_ridge)
r2 = r2_score(y_test, y_test_pred_ridge)

print("Mean Squared Error for test:", mse)
print("R2 Score for test:", r2)
cv_scores = cross_val_score(ridge, X_train, y_train, cv=5, scoring='r2')
print(f"CV scores:{cv_scores}")
print(f"Average CV score:{cv_scores.mean()}")

#### Hyperparameter Tuning

In [ ]:
param_grid = {
    'alpha': np.logspace(-4, 3, 50)   # 0.0001 to 1000
}

grid = GridSearchCV(estimator=Ridge(), param_grid=param_grid, scoring='r2', cv=5, n_jobs=-1, verbose=1)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV R2 score:", grid.best_score_)

best_ridge = grid.best_estimator_

y_train_pred_ridge = best_ridge.predict(X_train)
y_test_pred_ridge = best_ridge.predict(X_test)

print("Train R2:", r2_score(y_train, y_train_pred_ridge))
print("Test R2:", r2_score(y_test, y_test_pred_ridge))
print("Train MSE:", mean_squared_error(y_train, y_train_pred_ridge))
print("Test MSE:", mean_squared_error(y_test, y_test_pred_ridge))

#### Actual v/s Predicted

In [ ]:
plt.figure(figsize=(7, 6))

plt.scatter(y_test, y_test_pred_ridge, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], 
         [y_test.min(), y_test.max()], 
         linestyle='--')

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted (Ridge Regression)")
plt.show()

This model is a good model, it captures the trend well but withsome moderate errors.

### Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor()
dt.fit(X_train, y_train)

y_train_pred_dt = dt.predict(X_train)
y_test_pred_dt = dt.predict(X_test)

train_acc = r2_score(y_train, y_train_pred_dt)
test_acc = r2_score(y_test, y_test_pred_dt)

print(f"r2 score for train: {train_acc}")
print(f"r2 score for test: {test_acc}")
print(f"mse for train: {mean_squared_error(y_train, y_train_pred_dt)}")
print(f"mse for test: {mean_squared_error(y_test, y_test_pred_dt)}")
cv_scores = cross_val_score(dt, X_train, y_train, cv=5, scoring='r2')
print(f"CV scores:{cv_scores}")
print(f"Average CV score:{cv_scores.mean()}")

#### Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'max_depth': [None] + list(np.arange(1, 20)), 
    'min_samples_split': np.arange(2, 50),       
    'min_samples_leaf': np.arange(1, 25),        
    'max_features': ['auto', 'sqrt', 'log2', None] 
}

dt = DecisionTreeRegressor(random_state=42)

rscv = RandomizedSearchCV(estimator=dt, param_distributions=param_dist, n_iter=50,  cv=5, scoring='r2')

rscv.fit(X_train, y_train)

print(f"Best parameters: {rscv.best_params_}")
print(f"Best cross-validation R2 score: {rscv.best_score_}")
best_dt_model = rscv.best_estimator_
y_train_pred_dt = best_dt_model.predict(X_train)
y_test_pred_dt = best_dt_model.predict(X_test)
train_r2 = best_dt_model.score(X_train, y_train)
test_r2 = best_dt_model.score(X_test, y_test)

print(f"Train R2: {train_r2}")
print(f"Test R2: {test_r2}")

#### Hyperparameter Tuning

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_test_pred_dt, alpha=0.6)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)],
         linestyle='--')

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Decision Tree Regression\nActual vs Predicted (Tuned Model)")
plt.show()

It performs reasonably well but not it is the best model for this dataset.

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor()
rf.fit(X_train, y_train)

y_train_pred_rf = rf.predict(X_train)
y_test_pred_rf = rf.predict(X_test)

train_acc = r2_score(y_train, y_train_pred_rf)
test_acc = r2_score(y_test, y_test_pred_rf)

print(f"r2 score for train: {train_acc}")
print(f"r2 score for test: {test_acc}")
print(f"mse for train: {mean_squared_error(y_train, y_train_pred_rf)}")
print(f"mse for test: {mean_squared_error(y_test, y_test_pred_rf)}")
cv_scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='r2')
print(f"CV scores:{cv_scores}")
print(f"Average CV score:{cv_scores.mean()}")

#### Hyperparameter Tuning

In [ ]:
rf_param_dist = {'n_estimators': np.arange(50, 500, 50), 'max_depth': [None] + list(np.arange(3, 15)),
                 'min_samples_split': np.arange(2, 20), 'min_samples_leaf': np.arange(1, 10),
                 'max_features': ['sqrt', 'log2', 1.0],'bootstrap': [True, False]}

rf = RandomForestRegressor(random_state=42)

rscv = RandomizedSearchCV(estimator=rf, param_distributions=rf_param_dist, n_iter=100, cv=5, scoring='r2', verbose=1, random_state=42, n_jobs=-1)

rscv.fit(X_train, y_train)

print(f"Best RF parameters: {rscv.best_params_}")
print(f"Best RF cross-validation R2 score: {rscv.best_score_}")
best_rf_model = rscv.best_estimator_

train_r2 = best_rf_model.score(X_train, y_train)
test_r2 = best_rf_model.score(X_test, y_test)

y_test_pred_rf = best_rf_model.predict(X_test)
y_train_pred_rf = best_rf_model.predict(X_train)
test_rmse = mean_squared_error(y_test, test_pred, squared=False)

print(f"Train R2: {train_r2}")
print(f"Test R2: {test_r2}")
print(f"Test RMSE: {test_rmse}")

#### Actual v/s Predicted

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_test_pred_rf, alpha=0.6)
plt.plot(
    [min(y_test), max(y_test)],
    [min(y_test), max(y_test)],
    linestyle='--'
)

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Random Forest Regression\nActual vs Predicted (Tuned Model)")
plt.grid(True)
plt.show()

It performs very well and also captures most of the variations in the data and make accurate predictions overall. It is a good and reliable model for this dataset.

#### Learning Curve

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(estimator=best_rf_model , X=X_train, y=y_train, cv=5, scoring='r2', train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1, verbose=1)


train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)


plt.plot(train_sizes, train_mean, marker='o', label='Training R2')
plt.plot(train_sizes, test_mean, marker='s', label='Validation R2')

plt.title("Learning Curve for Random Forest")
plt.xlabel("Training Samples")
plt.ylabel("R2 Score")
plt.grid(True)
plt.legend()
plt.show()

### SVR

In [ ]:
from sklearn.svm import SVR

svr = SVR()
svr.fit(X_train, y_train)

y_train_pred_svr = svr.predict(X_train)
y_test_pred_svr = svr.predict(X_test)

train_acc = r2_score(y_train, y_train_pred_svr)
test_acc = r2_score(y_test, y_test_pred_svr)

print(f"r2 score for train: {train_acc}")
print(f"r2 score for test: {test_acc}")
print(f"mse for train: {mean_squared_error(y_train, y_train_pred_svr)}")
print(f"mse for test: {mean_squared_error(y_test, y_test_pred_svr)}")
cv_scores = cross_val_score(svr, X_train, y_train, cv=5, scoring='r2')
print(f"CV scores:{cv_scores}")
print(f"Average CV score:{cv_scores.mean()}")

#### Hyperparameter Tuning

In [ ]:
from scipy.stats import reciprocal, uniform

svr_param_dist = {
    'kernel': ['rbf', 'linear', 'poly', 'sigmoid'],
    'C': reciprocal(0.1, 1000),  
    'gamma': reciprocal(0.001, 1),
    'epsilon': uniform(0.01, 1)
}

svr = SVR()

svr_random_search = RandomizedSearchCV(estimator=svr, param_distributions=svr_param_dist, n_iter=50, cv=5, scoring='r2', verbose=1, random_state=42, n_jobs=-1)

svr_random_search.fit(X_train, y_train) 

print(f"Best SVR parameters: {svr_random_search.best_params_}")
print(f"Best SVR R2 score: {svr_random_search.best_score_}")
best_svr_model = svr_random_search.best_estimator_

train_r2 = best_svr_model.score(X_train, y_train)
test_r2 = best_svr_model.score(X_test, y_test)
y_test_pred_svr = best_svr_model.predict(X_test)
y_train_pred_svr = best_svr_model.predict(X_train)
test_rmse = mean_squared_error(y_test, y_test_pred_svr, squared=False)
print(f"SVR Train R2: {train_r2}")
print(f"SVR Test R2: {test_r2}")
print(f"SVR Test RMSE: {test_rmse}")

#### Predicted v/s Actual

In [ ]:
plt.scatter(y_test, y_test_pred_svr)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()])
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("SVR- Predicted vs Actual")
plt.show()

#### Learning Curve

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', best_svr_model)
])

train_sizes, train_scores, test_scores = learning_curve(estimator=pipeline, X=X_train, y=y_train, cv=5, scoring='r2', train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1, verbose=1)

train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.figure(figsize=(8,5))
plt.plot(train_sizes, train_mean, marker='o', label='Training R2')
plt.plot(train_sizes, test_mean, marker='s', label='Validation R2')
plt.title("Learning Curve of Tuned SVR Model")
plt.xlabel("Training Samples")
plt.ylabel("R2 Score")
plt.grid(True)
plt.legend()
plt.show()

### Gradient Boosting Regressor

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor()
gbr.fit(X_train, y_train)

y_train_pred_gbr = gbr.predict(X_train)
y_test_pred_gbr = gbr.predict(X_test)

train_acc = r2_score(y_train, y_train_pred_gbr)
test_acc = r2_score(y_test, y_test_pred_gbr)

print(f"r2 score for train: {train_acc}")
print(f"r2 score for test: {test_acc}")
print(f"mse for train: {mean_squared_error(y_train, y_train_pred_gbr)}")
print(f"mse for test: {mean_squared_error(y_test, y_test_pred_gbr)}")
cv_scores = cross_val_score(gbr, X_train, y_train, cv=5, scoring='r2')
print(f"CV scores:{cv_scores}")
print(f"Average CV score:{cv_scores.mean()}")

#### Hyperparameter Tuning

In [ ]:
gbr_param_dist = {
    'n_estimators': np.arange(100, 1000, 100),
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3], 
    'max_depth': np.arange(3, 8),
    'min_samples_split': np.arange(2, 20),
    'min_samples_leaf': np.arange(1, 10),
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'max_features': ['sqrt', 'log2', None]
}
gbr = GradientBoostingRegressor(random_state=42)

gbr_random_search = RandomizedSearchCV(estimator=gbr, param_distributions=gbr_param_dist, n_iter=50, cv=5, scoring='r2', random_state=42, n_jobs=-1)

gbr_random_search.fit(X_train, y_train)

print(f"Best GBR parameters: {gbr_random_search.best_params_}")
print(f"Best GBR R2 score: {gbr_random_search.best_score_}")
best_gbr_model = gbr_random_search.best_estimator_

y_test_pred_gbr = best_gbr_model.predict(X_test)
y_train_predi_gbr = best_gbr_model.predict(X_train)
tuned_r2 = r2_score(y_test, y_test_pred_gbr)
tuned_rmse = np.sqrt(mean_squared_error(y_test,y_test_pred_gbr ))

print(f"GBR Test R2: {tuned_r2}")
print(f"GBR Test RMSE: {tuned_rmse}")

#### Actual v/s Predicted

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_test_pred_gbr)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Actual")
plt.ylabel("Predicted Values")
plt.title("Tuned GradientBoostingRegressor: True vs Predicted")
plt.show()

This model works extremely well, the model fits the data almost perfectly.

### Adaboost

In [ ]:
from sklearn.ensemble import AdaBoostRegressor

ada = AdaBoostRegressor(
    n_estimators=100, 
    learning_rate=0.1,
    random_state=42
)

ada.fit(X_train, y_train)

y_train_pred_abr = ada.predict(X_train)
y_test_pred_abr = ada.predict(X_test)


train_r2 = r2_score(y_train, y_train_pred_abr)
test_r2 = r2_score(y_test, y_test_pred_abr)

train_mse = mean_squared_error(y_train, y_train_pred_abr)
test_mse = mean_squared_error(y_test, y_test_pred_abr)

print("Train R²:", train_r2)
print("Test R²:", test_r2)
print("Train MSE:", train_mse)
print("Test MSE:", test_mse)

#### Hyperparameter Tuning

In [ ]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

ada = AdaBoostRegressor(random_state=42)

param_dist = {
    'n_estimators': np.arange(50, 500, 50),
    'learning_rate': np.linspace(0.01, 1.0, 20),
    'loss': ['linear', 'square', 'exponential']
}

ada_random = RandomizedSearchCV(
    estimator=ada,
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

ada_random.fit(X_train, y_train)

print("Best Params:", ada_random.best_params_)
print("Best CV R2:", ada_random.best_score_)

best_ada = ada_random.best_estimator_

y_test_pred_abr = best_ada.predict(X_test)
y_train_pred_abr = best_ada.predict(X_train)
test_r2 = r2_score(y_test, y_test_pred_abr)
test_rmse = mean_squared_error(y_test, y_test_pred_abr, squared=False)

print("Test R2 (Tuned):", test_r2)
print("Test RMSE (Tuned):", test_rmse)

#### Actual v/s Predicted

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_test_pred_abr)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Tuned AdaBoost Regressor: True vs Predicted")
plt.show()

It also gives good performance and captures most of the patterns in the data but it is not that of a strong model.

### Important Features

Since SVR does so well (as per R2 Score), we try to see which features are most important with permutation_importance.

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(best_svr_model, X_test, y_test, scoring='r2', n_repeats=10, random_state=42)

importances = result.importances_mean
indices = np.argsort(importances)[::-1]

print("Feature importances:")
for i in indices:
    print(f"{X_train.columns[i]}: {importances[i]}")

plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), X_train.columns[indices], rotation=90)
plt.xlabel('Feature')
plt.ylabel('Feature importance')
plt.title("Permutation Feature Importance - SVR")
plt.show()

We calculate the feature importances with sklearn's feature_importance_ attribute for random forests as well.

In [ ]:
feat_imp = best_rf_model.feature_importances_
feat_ind = np.argsort(feat_imp)[::-1]
plt.bar(X_train.columns[feat_ind], feat_imp[feat_ind])
plt.xticks(range(len(feat_imp)), X_train.columns[feat_ind], rotation=90)
plt.xlabel('Feature')
plt.ylabel('Feature importance')
plt.title("Feature Importance - RF")
plt.show()

In [ ]:
result = permutation_importance(best_poly_model, X_test, y_test, scoring='r2', n_repeats=10, random_state=42)

importances = result.importances_mean
indices = np.argsort(importances)[::-1]

print("Feature importances:")
for i in indices:
    print(f"{X_train.columns[i]}: {importances[i]}")

plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), X_train.columns[indices], rotation=90)
plt.xlabel('Feature')
plt.ylabel('Feature importance')
plt.title("Permutation Feature Importance - Polynomial Regression")
plt.show()

All the top models indicate that the most important features that determine a candidate's exam score are Attendance and Hours studied. All the other features contribute to the exam score much less than these do.

In [ ]:
combined_results = pd.DataFrame({
    "Model": ["Linear Regression", "Polynomial Regression","LASSO","Ridge","DecisionTreeRegressor","RandomForestRegressor","SVR","GradientBoostingRegressor","AdaBoostRegressor"],
    "Train R2": [
        r2_score(y_train, y_train_pred_linear), 
        r2_score(y_train, y_train_pred_poly),
        r2_score(y_train, y_train_pred_lasso),
        r2_score(y_train, y_train_pred_ridge),
        r2_score(y_train, y_train_pred_dt),
        r2_score(y_train, y_train_pred_rf),
        r2_score(y_train, y_train_pred_svr),
        r2_score(y_train, y_train_pred_gbr),
        r2_score(y_train, y_train_pred_abr)
    ],
    "Test R2": [
        r2_score(y_test, y_test_pred_linear),
        r2_score(y_test, y_test_pred_poly),
        r2_score(y_test, y_test_pred_lasso),
        r2_score(y_test, y_test_pred_ridge),
        r2_score(y_test, y_test_pred_dt),
        r2_score(y_test, y_test_pred_rf),
        r2_score(y_test, y_test_pred_svr),
        r2_score(y_test, y_test_pred_gbr),
        r2_score(y_test, y_test_pred_abr)
    ],
    "Train MSE": [
        mean_squared_error(y_train, y_train_pred_linear),
        mean_squared_error(y_train, y_train_pred_poly),
        mean_squared_error(y_train, y_train_pred_lasso),
        mean_squared_error(y_train, y_train_pred_ridge),
        mean_squared_error(y_train, y_train_pred_dt),
        mean_squared_error(y_train, y_train_pred_rf),
        mean_squared_error(y_train, y_train_pred_svr),
        mean_squared_error(y_train, y_train_pred_gbr),
        mean_squared_error(y_train, y_train_pred_abr)
    ],
    "Test MSE": [
        mean_squared_error(y_test, y_test_pred_linear),
        mean_squared_error(y_test, y_test_pred_poly),
        mean_squared_error(y_test, y_test_pred_lasso),
        mean_squared_error(y_test, y_test_pred_ridge), 
        mean_squared_error(y_test, y_test_pred_dt),
        mean_squared_error(y_test, y_test_pred_rf),
        mean_squared_error(y_test, y_test_pred_svr),
        mean_squared_error(y_test, y_test_pred_gbr),
        mean_squared_error(y_test, y_test_pred_abr)
    ]
})

combined_results